# Projekt (eA): ein Lernverfahren anwenden

Informatik 13 · Künstliche Intelligenz · edu-mrh.de · CC BY-NC-SA 4.0

Dieselben Programme wie auf der Website p-projekt.html. Führe zuerst die Hilfszelle aus. Einrichtung von Python und Jupyter: https://edu-mrh.de/informatik/13/ki/python-einrichten.html

In [ ]:
# Hilfszelle: zuerst ausführen.
# Auf der Website zeichnet das Modul "ki" die Diagramme im Browser. Hier übernimmt das matplotlib.
# Datendateien (z. B. ring.csv) werden beim ersten Mal von edu-mrh.de geladen und neben dem Notebook gespeichert.
import os, urllib.request
import matplotlib.pyplot as plt

DATEN_URL = "https://edu-mrh.de/informatik/13/ki/data/"

def datei(name):
    if not os.path.exists(name):
        urllib.request.urlretrieve(DATEN_URL + name, name)
    return name

class ki:
    @staticmethod
    def linie(x, reihen, titel="", xname="", yname=""):
        if not isinstance(reihen, dict):
            reihen = {yname or "Wert": list(reihen)}
        for name, werte in reihen.items():
            plt.plot(list(x), werte, marker="o" if len(werte) < 40 else None, label=name)
        if len(reihen) > 1:
            plt.legend()
        plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.grid(alpha=.3); plt.show()

    @staticmethod
    def streu(punkte, klassen=None, titel="", xname="x1", yname="x2"):
        klassen = klassen or [0] * len(punkte)
        plt.scatter([p[0] for p in punkte], [p[1] for p in punkte], c=klassen, cmap="coolwarm", s=18)
        plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.gca().set_aspect("equal"); plt.show()

    @staticmethod
    def region(f, punkte=(), klassen=None, schritte=40, titel="", xname="x1", yname="x2", bereich=(0, 1)):
        a, b = bereich
        gitter = [[f(a + (b - a) * (i + 0.5) / schritte, a + (b - a) * (j + 0.5) / schritte) for i in range(schritte)] for j in range(schritte)]
        plt.imshow(gitter, origin="lower", extent=(a, b, a, b), cmap="coolwarm", alpha=.35, vmin=0, vmax=1)
        if punkte:
            plt.scatter([p[0] for p in punkte], [p[1] for p in punkte], c=klassen, cmap="coolwarm", s=18)
        plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.show()

    @staticmethod
    def balken(namen, werte, titel="", xname="", yname=""):
        plt.bar([str(n) for n in namen], werte); plt.title(titel); plt.xlabel(xname); plt.ylabel(yname); plt.show()

    @staticmethod
    def tabelle(zeilen, kopf=None, titel=""):
        if titel:
            print(titel)
        if kopf:
            print(" | ".join(str(k) for k in kopf))
        for z in zeilen:
            print(" | ".join(str(round(v, 4)) if isinstance(v, float) else str(v) for v in z))

    @staticmethod
    def ziffern(bilder, namen=None, titel=""):
        fig, achsen = plt.subplots(1, len(bilder), figsize=(1.2 * len(bilder), 1.6))
        if len(bilder) == 1:
            achsen = [achsen]
        for k, (ax, bild) in enumerate(zip(achsen, bilder)):
            ax.imshow([bild[i * 8:(i + 1) * 8] for i in range(8)], cmap="gray_r", vmin=0, vmax=16)
            ax.axis("off")
            if namen:
                ax.set_title(str(namen[k]), fontsize=9)
        fig.suptitle(titel); plt.show()

for _name in ["pinguine.csv", "ziffern.csv", "ring.csv"]:
    try:
        datei(_name)
    except Exception as _fehler:
        print("Konnte", _name, "nicht laden:", _fehler)
print("Hilfszelle bereit.")


## Projekt 1: k-NN mit verschiedenen Merkmalen

**Idee:** Für verschiedene Merkmalskombinationen und k wird gemessen, wie oft k-NN auf den Testdaten die richtige Art vorhersagt.
                            
1. Oben steht der Werkzeugkasten (fertig), unten das Experiment.
1. `laden` liest die gewünschten Spalten, `normieren` und `aufteilen` bereiten vor, `knn` sagt voraus.
                            

**Achte auf:** Wie viel bringt jedes weitere Merkmal? Wiederhole mit `seed=2` und `seed=3`: Wie stark schwanken die Werte?

In [ ]:
# ================= Werkzeugkasten (nicht verändern, nur benutzen) =================
import csv, math, random

def laden(datei, merkmale, ziel=None, nur=None):
    """Liest eine CSV-Datei. merkmale: Liste von Spaltennamen (Zahlen), ziel: Spalte mit der Klasse oder None.
    nur: z. B. ("art", ["Adelie", "Esels"]) behält nur diese Zeilen. Unvollständige Zeilen werden übersprungen."""
    daten = []
    for z in csv.DictReader(open(datei)):
        if nur and z[nur[0]] not in nur[1]:
            continue
        if any(z[m] in ("", "NA") for m in merkmale) or (ziel and z[ziel] in ("", "NA")):
            continue
        daten.append(([float(z[m]) for m in merkmale], z[ziel] if ziel else None))
    return daten

def normieren(daten):
    """Min-Max-Normierung jedes Merkmals auf 0 bis 1 (Station 2)."""
    n = len(daten[0][0])
    lo = [min(x[i] for x, _ in daten) for i in range(n)]
    hi = [max(x[i] for x, _ in daten) for i in range(n)]
    return [([(x[i] - lo[i]) / (hi[i] - lo[i]) if hi[i] > lo[i] else 0 for i in range(n)], k) for x, k in daten]

def aufteilen(daten, anteil_training=0.7, seed=1):
    """Mischt und teilt in Trainings- und Testdaten (Station 6)."""
    d = daten[:]
    random.Random(seed).shuffle(d)
    n = int(len(d) * anteil_training)
    return d[:n], d[n:]

def abstand(a, b):
    return math.sqrt(sum((p - q) ** 2 for p, q in zip(a, b)))

def knn(training, x, k=3):
    """k-nächste Nachbarn (Station 2): Mehrheit der k nächsten Trainingspunkte."""
    nah = sorted(training, key=lambda p: abstand(p[0], x))[:k]
    stimmen = {}
    for _, klasse in nah:
        stimmen[klasse] = stimmen.get(klasse, 0) + 1
    return max(stimmen, key=lambda c: (stimmen[c], -[kl for _, kl in nah].index(c)))

def kmeans(daten, k, seed=1, runden=100):
    """k-Means (Station 3). Gibt (Zuordnung je Punkt, Zentren, Streuung) zurück."""
    punkte = [x for x, _ in daten]
    zentren = [p[:] for p in random.Random(seed).sample(punkte, k)]
    zuordnung = None
    for _ in range(runden):
        neu = [min(range(k), key=lambda c: abstand(p, zentren[c])) for p in punkte]
        if neu == zuordnung:
            break
        zuordnung = neu
        for c in range(k):
            mitglieder = [p for p, a in zip(punkte, zuordnung) if a == c]
            if mitglieder:
                zentren[c] = [sum(v) / len(mitglieder) for v in zip(*mitglieder)]
    streuung = sum(abstand(p, zentren[a]) ** 2 for p, a in zip(punkte, zuordnung))
    return zuordnung, zentren, streuung

def genauigkeit(vorhersagen, wahr):
    return sum(1 for v, w in zip(vorhersagen, wahr) if v == w) / len(wahr)

def verwechslung(vorhersagen, wahr):
    """Verwechslungstabelle als Zeilen: [wahre Klasse, vorhergesagt, Anzahl]."""
    z = {}
    for v, w in zip(vorhersagen, wahr):
        z[(w, v)] = z.get((w, v), 0) + 1
    return [[w, v, n] for (w, v), n in sorted(z.items())]
# ===================================================================================

# ---------- Projekt 1: Welche Merkmale verraten die Art? ----------
versuche = [["schnabellaenge"], ["flossenlaenge"], ["schnabellaenge", "schnabeltiefe"], ["schnabeltiefe", "flossenlaenge"],
            ["schnabellaenge", "schnabeltiefe", "flossenlaenge"], ["schnabellaenge", "schnabeltiefe", "flossenlaenge", "gewicht"]]
zeilen = []
for merkmale in versuche:
    daten = normieren(laden("pinguine.csv", merkmale, ziel="art"))
    training, test = aufteilen(daten, 0.7, seed=1)
    for k in (1, 5):
        vorhersagen = [knn(training, x, k) for x, _ in test]
        zeilen.append([", ".join(merkmale), k, round(100 * genauigkeit(vorhersagen, [a for _, a in test]), 1)])
ki.tabelle(zeilen, ["Merkmale", "k", "richtig auf Testdaten (%)"], titel="k-NN mit verschiedenen Merkmalen")

## Projekt 2: Findet k-Means die Geschlechter?

**Idee:** k-Means bekommt nur Gewicht und Schnabeltiefe der Pinguine einer Art, keine Etiketten. Danach vergleichen wir die zwei Cluster mit dem Geschlecht.
                            

**Achte auf:** Wie viele Tiere landen „richtig“? Ändere `ART` auf "Adelie" und "Zuegel", probiere andere Merkmale und k = 3. Was heißt es, wenn ein Cluster einer Eigenschaft entspricht, die das Verfahren nie gesehen hat?

In [ ]:
# ================= Werkzeugkasten (nicht verändern, nur benutzen) =================
import csv, math, random

def laden(datei, merkmale, ziel=None, nur=None):
    """Liest eine CSV-Datei. merkmale: Liste von Spaltennamen (Zahlen), ziel: Spalte mit der Klasse oder None.
    nur: z. B. ("art", ["Adelie", "Esels"]) behält nur diese Zeilen. Unvollständige Zeilen werden übersprungen."""
    daten = []
    for z in csv.DictReader(open(datei)):
        if nur and z[nur[0]] not in nur[1]:
            continue
        if any(z[m] in ("", "NA") for m in merkmale) or (ziel and z[ziel] in ("", "NA")):
            continue
        daten.append(([float(z[m]) for m in merkmale], z[ziel] if ziel else None))
    return daten

def normieren(daten):
    """Min-Max-Normierung jedes Merkmals auf 0 bis 1 (Station 2)."""
    n = len(daten[0][0])
    lo = [min(x[i] for x, _ in daten) for i in range(n)]
    hi = [max(x[i] for x, _ in daten) for i in range(n)]
    return [([(x[i] - lo[i]) / (hi[i] - lo[i]) if hi[i] > lo[i] else 0 for i in range(n)], k) for x, k in daten]

def aufteilen(daten, anteil_training=0.7, seed=1):
    """Mischt und teilt in Trainings- und Testdaten (Station 6)."""
    d = daten[:]
    random.Random(seed).shuffle(d)
    n = int(len(d) * anteil_training)
    return d[:n], d[n:]

def abstand(a, b):
    return math.sqrt(sum((p - q) ** 2 for p, q in zip(a, b)))

def knn(training, x, k=3):
    """k-nächste Nachbarn (Station 2): Mehrheit der k nächsten Trainingspunkte."""
    nah = sorted(training, key=lambda p: abstand(p[0], x))[:k]
    stimmen = {}
    for _, klasse in nah:
        stimmen[klasse] = stimmen.get(klasse, 0) + 1
    return max(stimmen, key=lambda c: (stimmen[c], -[kl for _, kl in nah].index(c)))

def kmeans(daten, k, seed=1, runden=100):
    """k-Means (Station 3). Gibt (Zuordnung je Punkt, Zentren, Streuung) zurück."""
    punkte = [x for x, _ in daten]
    zentren = [p[:] for p in random.Random(seed).sample(punkte, k)]
    zuordnung = None
    for _ in range(runden):
        neu = [min(range(k), key=lambda c: abstand(p, zentren[c])) for p in punkte]
        if neu == zuordnung:
            break
        zuordnung = neu
        for c in range(k):
            mitglieder = [p for p, a in zip(punkte, zuordnung) if a == c]
            if mitglieder:
                zentren[c] = [sum(v) / len(mitglieder) for v in zip(*mitglieder)]
    streuung = sum(abstand(p, zentren[a]) ** 2 for p, a in zip(punkte, zuordnung))
    return zuordnung, zentren, streuung

def genauigkeit(vorhersagen, wahr):
    return sum(1 for v, w in zip(vorhersagen, wahr) if v == w) / len(wahr)

def verwechslung(vorhersagen, wahr):
    """Verwechslungstabelle als Zeilen: [wahre Klasse, vorhergesagt, Anzahl]."""
    z = {}
    for v, w in zip(vorhersagen, wahr):
        z[(w, v)] = z.get((w, v), 0) + 1
    return [[w, v, n] for (w, v), n in sorted(z.items())]
# ===================================================================================

# ---------- Projekt 2: Findet k-Means die Geschlechter? ----------
ART = "Esels"
daten = normieren(laden("pinguine.csv", ["gewicht", "schnabeltiefe"], ziel="geschlecht", nur=("art", [ART])))
zuordnung, zentren, streuung = kmeans(daten, 2, seed=1)
tabelle = {}
for c, (_, g) in zip(zuordnung, daten):
    tabelle[(c, g)] = tabelle.get((c, g), 0) + 1
print(len(daten), "Pinguine der Art", ART, "  Streuung", round(streuung, 2))
ki.tabelle([[c, tabelle.get((c, "w"), 0), tabelle.get((c, "m"), 0)] for c in range(2)], ["Cluster", "weiblich", "männlich"])
ki.streu([x for x, _ in daten], zuordnung, titel="Cluster (Farbe)", xname="Gewicht (normiert)", yname="Schnabeltiefe (normiert)")

## Dein Experiment (Vorlage)

**Idee:** Vorlage für eigene Versuchsreihen: eine Größe (hier k) wird variiert, das Ergebnis als Diagramm gezeichnet, dazu die Verwechslungstabelle.
                            

**Achte auf:** Ändere nur den Teil unter „Dein eigenes Experiment“. Notiere jede Einstellung und jedes Ergebnis im Protokoll.

In [ ]:
# ================= Werkzeugkasten (nicht verändern, nur benutzen) =================
import csv, math, random

def laden(datei, merkmale, ziel=None, nur=None):
    """Liest eine CSV-Datei. merkmale: Liste von Spaltennamen (Zahlen), ziel: Spalte mit der Klasse oder None.
    nur: z. B. ("art", ["Adelie", "Esels"]) behält nur diese Zeilen. Unvollständige Zeilen werden übersprungen."""
    daten = []
    for z in csv.DictReader(open(datei)):
        if nur and z[nur[0]] not in nur[1]:
            continue
        if any(z[m] in ("", "NA") for m in merkmale) or (ziel and z[ziel] in ("", "NA")):
            continue
        daten.append(([float(z[m]) for m in merkmale], z[ziel] if ziel else None))
    return daten

def normieren(daten):
    """Min-Max-Normierung jedes Merkmals auf 0 bis 1 (Station 2)."""
    n = len(daten[0][0])
    lo = [min(x[i] for x, _ in daten) for i in range(n)]
    hi = [max(x[i] for x, _ in daten) for i in range(n)]
    return [([(x[i] - lo[i]) / (hi[i] - lo[i]) if hi[i] > lo[i] else 0 for i in range(n)], k) for x, k in daten]

def aufteilen(daten, anteil_training=0.7, seed=1):
    """Mischt und teilt in Trainings- und Testdaten (Station 6)."""
    d = daten[:]
    random.Random(seed).shuffle(d)
    n = int(len(d) * anteil_training)
    return d[:n], d[n:]

def abstand(a, b):
    return math.sqrt(sum((p - q) ** 2 for p, q in zip(a, b)))

def knn(training, x, k=3):
    """k-nächste Nachbarn (Station 2): Mehrheit der k nächsten Trainingspunkte."""
    nah = sorted(training, key=lambda p: abstand(p[0], x))[:k]
    stimmen = {}
    for _, klasse in nah:
        stimmen[klasse] = stimmen.get(klasse, 0) + 1
    return max(stimmen, key=lambda c: (stimmen[c], -[kl for _, kl in nah].index(c)))

def kmeans(daten, k, seed=1, runden=100):
    """k-Means (Station 3). Gibt (Zuordnung je Punkt, Zentren, Streuung) zurück."""
    punkte = [x for x, _ in daten]
    zentren = [p[:] for p in random.Random(seed).sample(punkte, k)]
    zuordnung = None
    for _ in range(runden):
        neu = [min(range(k), key=lambda c: abstand(p, zentren[c])) for p in punkte]
        if neu == zuordnung:
            break
        zuordnung = neu
        for c in range(k):
            mitglieder = [p for p, a in zip(punkte, zuordnung) if a == c]
            if mitglieder:
                zentren[c] = [sum(v) / len(mitglieder) for v in zip(*mitglieder)]
    streuung = sum(abstand(p, zentren[a]) ** 2 for p, a in zip(punkte, zuordnung))
    return zuordnung, zentren, streuung

def genauigkeit(vorhersagen, wahr):
    return sum(1 for v, w in zip(vorhersagen, wahr) if v == w) / len(wahr)

def verwechslung(vorhersagen, wahr):
    """Verwechslungstabelle als Zeilen: [wahre Klasse, vorhergesagt, Anzahl]."""
    z = {}
    for v, w in zip(vorhersagen, wahr):
        z[(w, v)] = z.get((w, v), 0) + 1
    return [[w, v, n] for (w, v), n in sorted(z.items())]
# ===================================================================================

# ---------- Dein eigenes Experiment ----------
# Beispiel: Wie hängt die Genauigkeit von k ab? Ändere Merkmale, k, Trainingsanteil oder seed.
daten = normieren(laden("pinguine.csv", ["schnabellaenge", "flossenlaenge"], ziel="art"))
training, test = aufteilen(daten, 0.7, seed=1)
ks = list(range(1, 26, 2))
werte = []
for k in ks:
    vorhersagen = [knn(training, x, k) for x, _ in test]
    werte.append(100 * genauigkeit(vorhersagen, [a for _, a in test]))
ki.linie(ks, werte, titel="Richtig auf Testdaten (%) in Abhängigkeit von k", xname="k", yname="%")
vorhersagen = [knn(training, x, 5) for x, _ in test]
ki.tabelle(verwechslung(vorhersagen, [a for _, a in test]), ["wahre Art", "vorhergesagt", "Anzahl"], titel="Verwechslungstabelle für k = 5")